# Part 3

In [ ]:
!pip install -q codecarbon transformers accelerate sentencepiece pandas openpyxl

In [ ]:
import os
import json
import time
from datetime import datetime

import torch
import pandas as pd
import csv

from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import EmissionsTracker

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# CONFIG
# =========================
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # change later if needed

NUM_RUNS = 3
CPU_WARMUP_SECS = 300   # 5 min
GPU_WARMUP_SECS = 300    # 1 min
COOLDOWN_SECS = 60      # 1 min
MEASURE_POWER_SECS = 1

MAIN_OUTPUT_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"

# Input should come from Part 2 cleaned + split output
# Update model tag if your Part 2 model changes
PART2_MODEL_TAG = "Qwen__Qwen2.5-VL-7B-Instruct"
REPORTS_DIR = f"/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part2_runs/{PART2_MODEL_TAG}/run_1/REPORTS_SPLIT/reports_txt"

MODEL_TAG = MODEL_NAME.replace("/", "__")
PART3_ROOT = os.path.join(MAIN_OUTPUT_DIR, "phase1_part3_runs", MODEL_TAG)

os.makedirs(PART3_ROOT, exist_ok=True)

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=MEASURE_POWER_SECS,
        log_level="warning",
    )

In [ ]:
setup_tracking_dir = os.path.join(PART3_ROOT, "setup_tracking")

setup_tracker = make_tracker(
    output_dir=setup_tracking_dir,
    project_name="phase1_part3_setup",
    output_file="phase1_part3_setup_emissions.csv"
)

print("PART 3 SETUP TRACKER START")
setup_tracker.start()
setup_t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

setup_emissions = setup_tracker.stop()
setup_duration = time.time() - setup_t0
print("PART 3 SETUP TRACKER STOP")

setup_summary = {
    "phase": "phase1_part3",
    "model_name": MODEL_NAME,
    "setup_duration_sec": setup_duration,
    "setup_emissions_kg": setup_emissions,
    "device": device,
    "input_reports_dir": REPORTS_DIR
}
save_json(setup_summary, os.path.join(setup_tracking_dir, "phase1_part3_setup_summary.json"))

print("Qwen model loaded:", MODEL_NAME)
setup_summary

In [ ]:
def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_part3_llm_once():
    messages = [{"role": "user", "content": "Say hello in one word."}]
    inp = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    _ = model.generate(
        input_ids=inp.input_ids,
        attention_mask=inp.attention_mask,
        max_new_tokens=8,
        do_sample=False
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

def run_part3_warmup():
    warmup_cpu(CPU_WARMUP_SECS)

    print(f"GPU/LLM warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()
    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_part3_llm_once()
        time.sleep(2)
    print("GPU/LLM warm-up finished.\n")

print("\n========== PART 3 WARM-UP START (EXCLUDED) ==========")
run_part3_warmup()
print("========== PART 3 WARM-UP STOP ==========\n")

In [ ]:
SYSTEM_PROMPT = """
You are a Senior QA Engineer.

Input: UML diagram descriptions (already extracted) for ONE software project.
The description may span multiple pages and multiple UML types (class/activity/sequence/etc.).

Your job:
- Understand the project from ALL pages together
- Generate COMPLETE and DETAILED test cases that cover ALL described elements
- Cover positive, negative, and edge cases
- If some small detail is missing, make a minimal reasonable assumption and proceed
  (do not get stuck; do not invent lots of new things)

STRICT RULES:
- Output PLAIN TEXT only (no JSON)
- Do NOT output placeholders like <...>, "string", "..."
- Do NOT repeat UML extraction templates or instructions
- Use concrete, realistic values when needed (e.g., valid/invalid inputs)
- Keep module names aligned with what is present (or infer minimally from flows)
"""

USER_PROMPT_TEMPLATE = """
You will receive the UML description of ONE project (all pages together).

Task:
1) First write PROJECT OVERVIEW (3–5 lines).
2) Then write TEST CASES.

UML DESCRIPTION (INPUT):
------------------------
{report_text}
------------------------

OUTPUT FORMAT (STRICT):

PROJECT OVERVIEW:
(3–5 lines summarizing the system)

----------------------------
TEST CASES
----------------------------

For EACH test case, use EXACTLY this structure:

Test Case ID: TC-<MODULE>-<NUMBER>
Title:
Module:
Source UML Pages:
Preconditions:
Test Data:
Test Steps:
Expected Result:

Rules:
- Use sequential numbering: 001, 002, 003...
- Make Module a meaningful word (no placeholders)
- Source UML Pages must be numbers like: 1 or 2,3
- Test Steps must be numbered 1., 2., 3....
- Cover everything described across pages (entities, flows, validations)
"""

def run_llm(system_prompt: str, user_prompt: str, max_new_tokens: int = 2500):
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_prompt}\n<|assistant|>\n"

    input_token_count = len(tokenizer.encode(prompt, add_special_tokens=False))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "<|assistant|>" in text:
        text = text.split("<|assistant|>", 1)[-1].strip()

    output_token_count = len(tokenizer.encode(text.strip(), add_special_tokens=False))
    return text.strip(), input_token_count, output_token_count

def parse_testcases_from_text(llm_output: str):
    """
    Parses the plain-text test cases generated by the LLM into structured rows.
    If any test case is incomplete/broken, it gets dropped.
    """
    lines = [ln.rstrip() for ln in llm_output.splitlines()]
    rows = []

    current = None
    collecting_steps = False

    def finalize(tc):
        required = ["Test Case ID", "Title", "Module", "Source UML Pages",
                    "Preconditions", "Test Data", "Test Steps", "Expected Result"]
        for k in required:
            if k not in tc or not str(tc[k]).strip():
                return None
        bad_tokens = ["<", ">", "string", "..."]
        blob = " ".join(str(tc[k]) for k in required).lower()
        if any(bt in blob for bt in bad_tokens):
            return None
        return tc

    for ln in lines:
        s = ln.strip()
        if not s:
            continue

        if s.startswith("Test Case ID:"):
            if current:
                fin = finalize(current)
                if fin:
                    rows.append(fin)
            current = {
                "Test Case ID": s.replace("Test Case ID:", "").strip(),
                "Title": "",
                "Module": "",
                "Source UML Pages": "",
                "Preconditions": "",
                "Test Data": "",
                "Test Steps": "",
                "Expected Result": ""
            }
            collecting_steps = False
            continue

        if not current:
            continue

        if s.startswith("Title:"):
            current["Title"] = s.replace("Title:", "").strip()
            collecting_steps = False
        elif s.startswith("Module:"):
            current["Module"] = s.replace("Module:", "").strip()
            collecting_steps = False
        elif s.startswith("Source UML Pages:"):
            current["Source UML Pages"] = s.replace("Source UML Pages:", "").strip()
            collecting_steps = False
        elif s.startswith("Preconditions:"):
            current["Preconditions"] = s.replace("Preconditions:", "").strip()
            collecting_steps = False
        elif s.startswith("Test Data:"):
            current["Test Data"] = s.replace("Test Data:", "").strip()
            collecting_steps = False
        elif s.startswith("Test Steps:"):
            current["Test Steps"] = s.replace("Test Steps:", "").strip()
            collecting_steps = True
        elif s.startswith("Expected Result:"):
            current["Expected Result"] = s.replace("Expected Result:", "").strip()
            collecting_steps = False
        else:
            if collecting_steps:
                current["Test Steps"] = (current["Test Steps"] + "\n" + s).strip()
            else:
                if current["Expected Result"] == "":
                    if current["Test Data"]:
                        current["Test Data"] = (current["Test Data"] + " " + s).strip()
                    else:
                        current["Preconditions"] = (current["Preconditions"] + " " + s).strip()

    if current:
        fin = finalize(current)
        if fin:
            rows.append(fin)

    return rows

In [ ]:
def run_part3_single(run_id: int):
    run_root = os.path.join(PART3_ROOT, f"run_{run_id}")
    os.makedirs(run_root, exist_ok=True)

    OUT_ROOT = os.path.join(run_root, "TESTCASES_LLM_ALL")
    TOKEN_CSV = os.path.join(run_root, "phase1_part3_token_counts.csv")
    TRACKING_DIR = os.path.join(run_root, "inference_tracking")

    os.makedirs(OUT_ROOT, exist_ok=True)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    tracker = make_tracker(
        output_dir=TRACKING_DIR,
        project_name="phase1_part3_inference",
        output_file=f"phase1_part3_run_{run_id}_inference_emissions.csv"
    )

    print(f"\nPART 3 RUN {run_id} INFERENCE TRACKER START")
    tracker.start()
    infer_t0 = time.time()

    report_files = sorted([f for f in os.listdir(REPORTS_DIR) if f.lower().endswith(".txt")])
    print("Found reports:", len(report_files))
    print("Example:", report_files[:5])

    summary = []
    token_rows = []

    for i, report_file in enumerate(report_files, start=1):
        report_path = os.path.join(REPORTS_DIR, report_file)
        report_name = os.path.splitext(report_file)[0]

        print(f"\n[{i}/{len(report_files)}] {report_file}")

        with open(report_path, "r", encoding="utf-8") as f:
            report_text = f.read().strip()

        if len(report_text) < 200:
            print("⚠️ Skipped (too short)")
            summary.append({"report": report_name, "test_cases": 0, "status": "skipped_short"})
            token_rows.append({
                "phase": "phase1_part3",
                "run_id": run_id,
                "report": report_name,
                "input_tokens": 0,
                "output_tokens": 0,
                "total_tokens": 0,
                "status": "skipped_short"
            })
            continue

        user_prompt = USER_PROMPT_TEMPLATE.format(report_text=report_text)
        llm_output, in_tok, out_tok = run_llm(SYSTEM_PROMPT, user_prompt, max_new_tokens=3000)

        rows = parse_testcases_from_text(llm_output)
        df = pd.DataFrame(rows)

        out_dir = os.path.join(OUT_ROOT, report_name)
        os.makedirs(out_dir, exist_ok=True)

        txt_out = os.path.join(out_dir, f"{report_name}_testcases.txt")
        with open(txt_out, "w", encoding="utf-8") as f:
            f.write(llm_output)

        csv_out = os.path.join(out_dir, f"{report_name}_testcases.csv")
        if len(df) > 0:
            with open(csv_out, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=df.columns, quoting=csv.QUOTE_ALL)
                writer.writeheader()
                for _, row in df.iterrows():
                    writer.writerow({k: str(v) for k, v in row.items()})

            xlsx_out = os.path.join(out_dir, f"{report_name}_testcases.xlsx")
            df.to_excel(xlsx_out, index=False)

            print(f"✅ Saved {len(df)} test cases → {out_dir}")
            summary.append({"report": report_name, "test_cases": len(df), "status": "ok"})
        else:
            print("⚠️ No valid test cases parsed (saved txt only)")
            summary.append({"report": report_name, "test_cases": 0, "status": "no_parsed_cases"})

        token_rows.append({
            "phase": "phase1_part3",
            "run_id": run_id,
            "report": report_name,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "total_tokens": (in_tok or 0) + (out_tok or 0),
            "status": summary[-1]["status"]
        })

    summary_df = pd.DataFrame(summary)
    summary_csv = os.path.join(OUT_ROOT, "SUMMARY.csv")
    summary_df.to_csv(summary_csv, index=False)

    token_df = pd.DataFrame(token_rows)
    token_df.to_csv(TOKEN_CSV, index=False)

    inference_emissions = tracker.stop()
    inference_duration = time.time() - infer_t0
    print(f"PART 3 RUN {run_id} INFERENCE TRACKER STOP")

    inference_summary = {
        "phase": "phase1_part3",
        "run_id": run_id,
        "mode": "repeated_runs",
        "model_name": MODEL_NAME,
        "input_reports_dir": REPORTS_DIR,
        "output_root": OUT_ROOT,
        "summary_csv": summary_csv,
        "token_csv": TOKEN_CSV,
        "inference_duration_sec": inference_duration,
        "inference_emissions_kg": inference_emissions,
    }

    save_json(
        inference_summary,
        os.path.join(TRACKING_DIR, f"phase1_part3_run_{run_id}_inference_summary.json")
    )

    print("\n✅ DONE. Summary saved:", summary_csv)

    return inference_summary

In [ ]:
all_run_summaries = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n==================== PART 3 RUN {run_id}/{NUM_RUNS} ====================")
    run_summary = run_part3_single(run_id)
    all_run_summaries.append(run_summary)

    if run_id < NUM_RUNS:
        print(f"\nCooldown for {COOLDOWN_SECS} seconds...\n")
        time.sleep(COOLDOWN_SECS)